# 0.0 — Adquisición de datos

**Objetivo.** Descargar, verificar y dejar en `data/raw/` los datasets crudos de YRBS y WONDER/NCHS. Documentar la procedencia con SHA-256 en `references/data_provenance.md` y dejar los datos limpios en formato Parquet en `data/processed/` para que el resto del pipeline no dependa de ODBC.

**Estructura del notebook.**
1. Setup y configuración de rutas.
2. Verificación de los archivos YRBS ya descargados (hashes, conteos).
3. Conversión de YRBS (.mdb → .parquet) usando `pyodbc` + `pandas`.
4. Descarga de datos de mortalidad adolescente (NCHS Socrata API).
5. Registro de provenance en `references/data_provenance.md`.


## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from datetime import datetime, timezone

import pandas as pd
import pyodbc
import requests

from wired_apart import config
from wired_apart.dataset import sha256_of

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 2. Verificación de YRBS

**Diagnóstico.** Verificamos que los 9 años de YRBS (2005-2021) estén presentes, calculamos SHA-256, contamos filas/columnas y confirmamos que las variables clave (`q25`, `q80`, `weight`) existan en cada año.

**Decisión.** Si todos los años pasan, procedemos a la conversión. Si alguno falta o no carga, paramos y reportamos.

In [ ]:
YEARS = [2005, 2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021]
verif = []
for year in YEARS:
    path = config.RAW_DIR / "yrbs" / f"yrbs{year}.mdb"
    if not path.exists():
        verif.append({"year": year, "status": "MISSING", "path": str(path)})
        continue
    digest = sha256_of(path)
    conn_str = r"Driver={Microsoft Access Driver (*.mdb, *.accdb)};DBQ=" + str(path) + ";"
    conn = pyodbc.connect(conn_str)
    tables = [t.table_name for t in conn.cursor().tables(tableType="TABLE")]
    tables = [t for t in tables if not t.startswith("MSys")]
    df = pd.read_sql(f"SELECT * FROM [{tables[0]}]", conn)
    conn.close()
    verif.append({
        "year": year,
        "status": "OK",
        "sha256_short": digest[:12],
        "size_mb": round(path.stat().st_size / 1e6, 1),
        "rows": len(df),
        "cols": df.shape[1],
        "has_q25": "q25" in df.columns,
        "has_q80": "q80" in df.columns,
        "has_weight": "weight" in df.columns,
    })
df_verif = pd.DataFrame(verif)
df_verif

**Verificación.** Todos los años deben tener `status=OK` y las tres variables clave presentes. Si alguna fila no cumple, revisar `data/raw/yrbs/` antes de continuar.

## 3. Conversión YRBS → Parquet

**Decisión.** Convertimos cada `.mdb` a `.parquet` con tipos de columna explícitos (float64 para q-vars y weights, preserva NaN) y agregamos una columna `year` para identificar el origen. Esto desacopla el resto del pipeline de `pyodbc`, que solo está disponible en Windows.

**Por qué Parquet y no CSV:** ~10x más pequeño en disco, tipos preservados, lectura columnar.

In [ ]:
INTERIM_DIR = config.INTERIM_DIR / "yrbs"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = config.PROCESSED_DIR
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for year in YEARS:
    path = config.RAW_DIR / "yrbs" / f"yrbs{year}.mdb"
    conn_str = r"Driver={Microsoft Access Driver (*.mdb, *.accdb)};DBQ=" + str(path) + ";"
    conn = pyodbc.connect(conn_str)
    df = pd.read_sql("SELECT * FROM [XXHq]", conn)
    conn.close()
    df["year"] = year
    q_cols = [c for c in df.columns if c.startswith("q") and c[1:].isdigit()]
    for c in q_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    if "weight" in df.columns:
        df["weight"] = pd.to_numeric(df["weight"], errors="coerce").astype("float64")
    if "stratum" in df.columns:
        df["stratum"] = pd.to_numeric(df["stratum"], errors="coerce").astype("float64")
    if "psu" in df.columns:
        df["psu"] = pd.to_numeric(df["psu"], errors="coerce").astype("float64")
    out = INTERIM_DIR / f"yrbs{year}.parquet"
    df.to_parquet(out, index=False)
    print(f"{year}: {len(df):>6,} rows -> {out.name} ({out.stat().st_size/1e6:.1f} MB)")

all_yrs = []
for year in YEARS:
    all_yrs.append(pd.read_parquet(INTERIM_DIR / f"yrbs{year}.parquet"))
df_all = pd.concat(all_yrs, ignore_index=True)
out_all = PROCESSED_DIR / "yrbs_2005_2021.parquet"
df_all.to_parquet(out_all, index=False)
print(f"\nStacked: {len(df_all):,} rows total -> {out_all.name} ({out_all.stat().st_size/1e6:.1f} MB)")
print(f"  -> columns: {df_all.shape[1]}")
print(f"  -> year range: {df_all['year'].min()}-{df_all['year'].max()}")

## 4. Datos de mortalidad adolescente (NCHS Socrata API)

**Estrategia.** Para el segundo dataset usamos la **API Socrata de NCHS en data.cdc.gov** (dataset `w26f-tf3h`), que provee tasas de suicidio por año/edad/sexo. Cobertura: **2018-2024** (no llegamos a 2005 con esta API).

Para los años 2005-2017, las opciones son:
1. WONDER API (XML específico + rate limits de 15s/request; complejo de automatizar)
2. NCHS Data Briefs publicados (PDFs oficiales con números anuales tabulados)
3. Tablas de mortalidad anuales NVSS

**Decisión operativa.** Descargamos el rango disponible de Socrata (2018-2024) ahora. Los años 2005-2017 se obtienen en una iteración posterior desde la UI de WONDER o scrapeando NCHS Data Briefs. Documentamos en `references/data_provenance.md`.

In [ ]:
SOCRATA_URL = "https://data.cdc.gov/resource/w26f-tf3h.json"
# 'group' es palabra reservada en SoQL -> backticks obligatorios
params = {
    "$where": "topic='Suicides' AND `group`='Sex and age group'",
    "$limit": 5000,
}
r = requests.get(SOCRATA_URL, params=params, timeout=60)
r.raise_for_status()
wonder = pd.DataFrame(r.json())
wonder = wonder[wonder["estimate_type"].str.contains("crude", case=False, na=False)].copy()
print(f"Rows from Socrata (crude, sex x age group): {len(wonder)}")
print(f"Years: {sorted(wonder['time_period'].unique()) if 'time_period' in wonder.columns else 'N/A'}")
print(f"Subgroups (sample): {sorted(wonder['subgroup'].unique())[:8] if 'subgroup' in wonder.columns else 'N/A'}")

In [ ]:
adolescent = wonder[wonder["subgroup"].str.contains(r"10-14|15-19", na=False)].copy()
adolescent = adolescent.rename(columns={
    "time_period": "year",
    "subgroup": "sex_age",
    "nesting_label": "sex",
    "estimate": "rate_per_100k",
    "standard_error": "se",
    "estimate_lci": "lci",
    "estimate_uci": "uci",
})
adolescent["year"] = adolescent["year"].astype(int)
for c in ["rate_per_100k", "se", "lci", "uci"]:
    adolescent[c] = pd.to_numeric(adolescent[c], errors="coerce")

out = PROCESSED_DIR / "wonder_suicide_adolescent_2018_2024.csv"
adolescent[["year", "sex_age", "sex", "rate_per_100k", "se", "lci", "uci", "estimate_type"]].to_csv(out, index=False)
print(f"Wrote {len(adolescent)} rows to {out.name}")
print(adolescent.head(10).to_string())

## 5. Registro de provenance

Documentamos cada archivo crudo con su URL, fecha, hash y notas.

In [ ]:
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
lines = [f"# Data Provenance — Wired Apart\n", f"_Last updated: {now}_\n"]

lines.append("## CDC YRBS (Youth Risk Behavior Surveillance System)\n\n")
lines.append("**Base URL:** https://www.cdc.gov/yrbs/data/index.html\n")
lines.append("**Format:** Microsoft Access (`.mdb`), with codebook PDFs.\n\n")

yrbs_urls = {
    2005: "https://www.cdc.gov/yrbs/files/2005/yrbs2005.zip",
    2007: "https://www.cdc.gov/yrbs/files/2007/yrbs2007.zip",
    2009: "https://www.cdc.gov/yrbs/files/2009/yrbs2009.zip",
    2011: "https://www.cdc.gov/yrbs/files/2011/yrbs2011.zip",
    2013: "https://www.cdc.gov/yrbs/files/2013/YRBS2013.zip",
    2015: "https://www.cdc.gov/yrbs/files/2015/yrbs2015.zip",
    2017: "https://www.cdc.gov/yrbs/files/2017/XXH2017_YRBS_Data.zip",
    2019: "https://www.cdc.gov/yrbs/files/2019/XXH2019_YRBS_Data.zip",
    2021: "https://www.cdc.gov/yrbs/files/2021/XXH2021_YRBS_Data.zip",
}
for year in YEARS:
    path = config.RAW_DIR / "yrbs" / f"yrbs{year}.mdb"
    codebook = config.RAW_DIR / "yrbs" / f"{year}_codebook.pdf"
    digest = sha256_of(path)
    size = path.stat().st_size
    lines.append(f"### yrbs{year}.mdb @ {now}\n\n")
    lines.append(f"- **URL:** {yrbs_urls[year]}\n")
    lines.append(f"- **Local path:** `data/raw/yrbs/yrbs{year}.mdb`\n")
    lines.append(f"- **SHA-256:** `{digest}`\n")
    lines.append(f"- **Size:** {size/1e6:.1f} MB\n")
    lines.append(f"- **Codebook:** `data/raw/yrbs/{year}_codebook.pdf` (SHA-256: `{sha256_of(codebook)}`)\n")
    lines.append("- **Variables clave:** q25 (sad/hopeless), q26-q28 (suicidalidad), q80 (screen time), weight, stratum, psu\n")
    lines.append("- **License:** Public domain (US Government work)\n\n")

lines.append("## NCHS Suicide Death Rates (data.cdc.gov Socrata API)\n\n")
lines.append(f"**URL:** {SOCRATA_URL}\n")
lines.append("**Dataset ID:** `w26f-tf3h`\n")
lines.append("**Coverage retrieved:** 2018-2024 (subset; full coverage in source)\n")
lines.append("**Filter applied:** `topic='Suicides' AND ` + BT + `group` + BT + `='Sex and age group'` (crude rate filtered in pandas)\n")
lines.append(f"**Retrieved at:** {now}\n")
lines.append("**Output:** `data/processed/wonder_suicide_adolescent_2018_2024.csv`\n\n")
lines.append("**Pendiente (Fase 3):** descargar 2005-2017 desde WONDER UI o NCHS Data Briefs para tener la serie completa pre/post Great Rewiring.\n")

provenance_path = config.REFERENCES_DIR / "data_provenance.md"
provenance_path.write_text("".join(lines), encoding="utf-8")
print(f"Wrote {provenance_path}")
print(f"Total bytes: {provenance_path.stat().st_size}")